# Linear Regression Example

This notebook demonstrates how to use the `LinearRegression` class from your `rice_ml` library.

It is written to work with your current repository structure:

- `src/rice_ml/supervised_ml/linear_regression.py`

To avoid the current `rice_ml/__init__.py` import issue, this notebook loads the class directly from the module file.


## 1. Import packages

We use NumPy, pandas, and matplotlib for data handling and visualization.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib.util

## 2. Locate the repository root and load `LinearRegression`

This cell searches upward until it finds your project root, then imports the class directly from:

`src/rice_ml/supervised_ml/linear_regression.py`


In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        target = path / "src" / "rice_ml" / "supervised_ml" / "linear_regression.py"
        if target.exists():
            return path
    raise FileNotFoundError(
        "Could not find repo root containing src/rice_ml/supervised_ml/linear_regression.py"
    )

repo_root = find_repo_root()
module_path = repo_root / "src" / "rice_ml" / "supervised_ml" / "linear_regression.py"

spec = importlib.util.spec_from_file_location("linear_regression_module", module_path)
linear_regression_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(linear_regression_module)

LinearRegression = linear_regression_module.LinearRegression

print("Repository root:", repo_root)
print("Loaded module from:", module_path)
print("Imported class:", LinearRegression)

## 3. Create a simple synthetic dataset

We generate one predictor `x` and one response `y` with a roughly linear relationship plus random noise.


In [ ]:
np.random.seed(42)

n = 200
x = np.random.uniform(0, 10, size=n)
noise = np.random.normal(0, 2.0, size=n)

y = 4.5 + 3.2 * x + noise

df = pd.DataFrame({"x": x, "y": y})
df.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["x"], df["y"], alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Linear Regression Data")
plt.show()

## 4. Train-test split

We split the data into a training set and a testing set.


In [ ]:
def train_test_split_numpy(X, y, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    rng.shuffle(indices)

    n_test = int(np.floor(test_size * n_samples))
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X = df[["x"]].to_numpy()
y_array = df["y"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y_array, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 5. Fit the linear regression model

Here we fit the default ordinary least squares model.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Coefficient(s):", model.coef_)

## 6. Make predictions


In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("First 5 test predictions:", y_test_pred[:5])

## 7. Evaluate the model

We check common regression metrics on both the training set and the testing set.


In [ ]:
results = pd.DataFrame({
    "Set": ["Train", "Test"],
    "R2": [model.score(X_train, y_train), model.score(X_test, y_test)],
    "MSE": [model.mse(X_train, y_train), model.mse(X_test, y_test)],
    "RMSE": [model.rmse(X_train, y_train), model.rmse(X_test, y_test)],
    "MAE": [model.mae(X_train, y_train), model.mae(X_test, y_test)],
})

results

## 8. Plot fitted line

Since this example has one predictor, we can visualize the fitted regression line directly.


In [ ]:
x_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_plot = model.predict(x_plot)

plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], y_train, alpha=0.6, label="Training data")
plt.scatter(X_test[:, 0], y_test, alpha=0.6, label="Testing data")
plt.plot(x_plot[:, 0], y_plot, linewidth=2, label="Fitted line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Fit")
plt.legend()
plt.show()

## 9. Residual diagnostics

A good regression fit should have residuals scattered around zero without a clear pattern.


In [ ]:
model.plot_residuals(X_test, y_test)

## 10. Model summary


In [ ]:
model.summary(X_test, y_test)

## 11. Optional: Ridge-style regularization

Your implementation also supports L2 regularization through the `regularization` argument.


In [ ]:
ridge_model = LinearRegression(regularization=1.0)
ridge_model.fit(X_train, y_train)

ridge_results = pd.DataFrame({
    "Model": ["OLS", "Ridge (lambda=1.0)"],
    "Intercept": [model.intercept_, ridge_model.intercept_],
    "Slope": [model.coef_[0], ridge_model.coef_[0]],
    "Test R2": [model.score(X_test, y_test), ridge_model.score(X_test, y_test)],
    "Test RMSE": [model.rmse(X_test, y_test), ridge_model.rmse(X_test, y_test)],
})

ridge_results

## 12. Optional: Gradient descent fitting

Your class can also fit the model using gradient descent.


In [ ]:
gd_model = LinearRegression(
    use_gradient_descent=True,
    learning_rate=0.001,
    max_iter=20000,
    tol=1e-8
)
gd_model.fit(X_train, y_train)

gd_results = pd.DataFrame({
    "Model": ["Closed-form OLS", "Gradient Descent"],
    "Intercept": [model.intercept_, gd_model.intercept_],
    "Slope": [model.coef_[0], gd_model.coef_[0]],
    "Test R2": [model.score(X_test, y_test), gd_model.score(X_test, y_test)],
    "Test RMSE": [model.rmse(X_test, y_test), gd_model.rmse(X_test, y_test)],
})

gd_results

## 13. Conclusion

In this notebook, we:

- loaded your custom `LinearRegression` implementation,
- generated a simple linear dataset,
- fit and evaluated the model,
- visualized the fitted line and residuals,
- and compared OLS, ridge-style regularization, and gradient descent.

You can place this notebook in:

`examples/Supervised_learning/linear_regression.ipynb`

inside your GitHub repository.
